# Police Station Spatial Intelligence — Community

## //00 Setup | Import Libraries
> All topologicpy modules needed for geometry, topology, and graph analysis.

In [1]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

## //01 Version Check
> Confirm topologicpy meets the minimum required version (0.9.31+).

In [2]:
print("This tutorial requires topologicpy version 0.9.31 or newer.")
print(Helper.Version())

This tutorial requires topologicpy version 0.9.18 or newer.
The version that you are using (0.9.29) is EQUAL TO the latest version available on PyPI.


## //02 Renderer | Configuration
> Set render target. Options: `vscode` | `colab` | `browser`.

In [3]:
renderer = "vscode"

## //03 Utility Functions
> `reset_dictionaries` — clear face metadata | `transfer_dicts_by_key` — propagate graph values back to geometry.

In [4]:
def reset_dictionaries(shell):
    faces = Topology.Faces(shell)
    for i, f in enumerate(faces):
        d = Topology.Dictionary(f)
        keys = Dictionary.Keys(d)
        for key in keys:
            if not key == "face_id":
                d = Dictionary.RemoveKey(d, key)
        f = Topology.SetDictionary(f, d)

def transfer_dicts_by_key(topologies, selectors, key):
    dicts = {}
    for t in topologies:
        d = Topology.Dictionary(t)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            dicts[str(value)] = t
    
    for s in selectors:
        d = Topology.Dictionary(s)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            f = dicts.get(str(value), None)
            if f:
                f = Topology.SetDictionary(f, d)


## //04 Import Floor Plan
> Load the processed ground-floor BREP face from `gf-floor-plan-face.brep`.

In [5]:
police_station = Topology.ByBREPPath(r"../assets/gf-floor-plan-face.brep")

## //05 Visualize Geometry
> Raw floor plan — single face, no grid.

In [6]:
Topology.Show(police_station,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor="white",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

## //06 Grid Overlay
> Compute bounding rectangle and generate a 2×2 unit edge grid clipped to the floor plan.

In [7]:
b_r = Wire.BoundingRectangle(police_station)
d = Topology.Dictionary(b_r)
xmin = Dictionary.ValueAtKey(d, "xmin")
xmax = Dictionary.ValueAtKey(d, "xmax")
ymin = Dictionary.ValueAtKey(d, "ymin")
ymax = Dictionary.ValueAtKey(d, "ymax")
width = Dictionary.ValueAtKey(d, "width")
length = Dictionary.ValueAtKey(d, "length")
uRange = list(range(0,int(width)+2,2))
vRange = list(range(0,int(length)+2,2))

grid = Grid.EdgesByDistances(police_station, clip=True, uRange=uRange, vRange=vRange)

## //07 Visualize Geometry | Grid

In [8]:
Topology.Show(police_station, grid,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor="grey",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,height=600,
              renderer = renderer)

## //08 Slice Floor Plan | Shell
> Divide the face into a regular grid of cells. Each cell becomes a graph node.

In [9]:
shell = Topology.Slice(police_station, grid)
faces = Topology.Faces(shell)
# Assign a sequential unique face id to reference it later (e.g. "face_21")
for i, f in enumerate(faces):
    d = Dictionary.ByKeyValue("face_id", "face_"+str(i+1))
    f = Topology.SetDictionary(f, d)

## //09 Derive Analysis Graph

In [10]:
# Note: Graph nodes automatically inherit the dictionaries of the entities they 
analysis_graph = Graph.ByTopology(shell)

## //10 Store Graph Vertices

In [11]:
g_verts = Graph.Vertices(analysis_graph)

## //11 Spatial Analysis

### //11a Community Detection
> Partitions the graph into spatially coherent clusters (~5 min).

In [12]:
community_list = Graph.CommunityPartition(analysis_graph, colorScale="thermal")

In [13]:
reset_dictionaries(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

In [14]:
Topology.Show(faces,
              faceColorKey="cp_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

### //11b Degree Centrality | Community-Scale

> Bin faces by community number | derive outer boundary per group | build community-scale shell.

In [15]:
bins = Topology.BinByDictionaryKey(faces, key="community")
bin_dict = bins[0]
keys = list(bin_dict.keys())
face_groups = []
for key in keys:
    bin_faces = bin_dict[key]
    temp_shell = Shell.ByFaces(bin_faces)
    eb = Shell.ExternalBoundary(temp_shell)
    eb = Wire.RemoveCollinearEdges(eb)
    eb = Face.ByWire(eb)
    face_groups.append(eb)



> Visualize community zones.

In [16]:
Topology.Show(face_groups,
              faceOpacity=1,
              showEdges=True,
              edgeWidth=8,
              edgeColor="grey",
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

> Build a new shell from the community face groups.

In [17]:
new_shell = Shell.ByFaces(face_groups)


> Visualize the community shell.

In [18]:
Topology.Show(new_shell,
              faceOpacity=0.9,
              showEdges=True,
              showVertices=True,
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

> Derive a new graph from the community shell.

In [19]:
new_graph = Graph.ByTopology(new_shell)
new_verts = Graph.Vertices(new_graph)
for v in new_verts:
    d = Dictionary.ByKeysValues(["color", "size"], ["red", 12])
    v = Topology.SetDictionary(v, d)

> Visualize the community graph.

In [20]:
Topology.Show(new_shell, new_graph,
              faceOpacity=0.9,
              showEdges=True,
              showVertices=True,
              vertexSizeKey="size",
              vertexColorKey="color",
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

> Compute degree centralities on the community graph.

In [21]:
degree_centralities = Graph.DegreeCentrality(new_graph, normalize=False)

> Interpolate degree centrality values from community vertices to original grid vertices.

In [22]:
for v in g_verts:
    new_v = Vertex.InterpolateValue(v, vertices=new_verts, n=3, key="degree_centrality")

> Derive vertex colors from interpolated degree centrality values.

In [23]:
minValue = min(degree_centralities)
maxValue = max(degree_centralities)
for v in g_verts:
    d = Topology.Dictionary(v)
    d_c = Dictionary.ValueAtKey(d, "degree_centrality")
    color = Color.AnyToHex(Color.ByValueInRange(d_c, minValue=minValue, maxValue=maxValue, colorScale="thermal"))
    d = Dictionary.SetValueAtKey(d, "dc_color", color)
    d = Dictionary.SetValueAtKey(d, "size", 16)
    v = Topology.SetDictionary(v, d)

> Transfer degree centrality from graph vertices to shell faces.

In [24]:
reset_dictionaries(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

> Visualize degree centrality on the floor plan.

In [25]:
Topology.Show(faces,
              faceColorKey="dc_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              vertexSizeKey="size",
              vertexColorKey="dc_color",
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)